In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd
import logging
import os

from IPython.display import display

from gsm_benchmarker.results_analyser import MultiVariantMultiModelResultsAnalyser

logger = logging.getLogger('notebook')

plt.style.use('default')
plt.style.use('seaborn-v0_8-muted')
plt.style.use('seaborn-v0_8-darkgrid')


In [ ]:
METRIC = "correct"

In [ ]:
pp = Path("../../../../data/gsm-symbolic/outputs").resolve()

p_standard_pilot = pp / "mini_20x50x4__14_11/final"
p_standard = pp / "default_full__06_02/final"

p_code = pp / 'code_full__09_02/corrected'
p_formal = pp / 'formalised_full__16_04/final'
p_nonformal = pp / 'nonformalised_full__20_04/final'

In [ ]:
# mres_standard_pilot = MultiVariantMultiModelResultsAnalyser(p_standard_pilot)

mres_standard = MultiVariantMultiModelResultsAnalyser(p_standard)

# mres_code = MultiVariantMultiModelResultsAnalyser(p_code)
# mres_formal = MultiVariantMultiModelResultsAnalyser(p_formal)
# mres_nonformal = MultiVariantMultiModelResultsAnalyser(p_nonformal)


In [ ]:
OUTPUTS = Path("./outputs").resolve()
os.makedirs(OUTPUTS, exist_ok=True)
OUTPUTS_FOLDER = str(OUTPUTS) + "/"

In [ ]:
mres_standard.models

In [ ]:
mres_standard.variants['main'].full_data

In [ ]:
import re

first_model = mres_standard.models[0]
number_pattern = re.compile(r'[+-]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?')
integer_bin_edges = [0, 1, 2, 3, 4, 5, 10, 20, 50, 100, 1000, float('inf')]
raw_integer_bin_labels = [f'[{integer_bin_edges[i]}, {integer_bin_edges[i+1]})' for i in range(len(integer_bin_edges) - 1)]
bin_labels = ['fractions']
for start, end in zip(integer_bin_edges[:-1], integer_bin_edges[1:]):
    if np.isinf(end):
        label = f"{start}+"
    elif (end - start) == 1:
        label = f"{start}"
    else:
	    label = f'{start}-{end}'
    bin_labels.append(label)

pretty_label_map = {label: str(i) for i, label in enumerate(raw_integer_bin_labels[:5])}
pretty_label_map.update({label: label for label in raw_integer_bin_labels[5:]})
variant_number_counts = {}
variant_raw_numbers = {}
for variant_name, analyser in mres_standard.variants.items():
    variant_df = analyser.full_data
    model_df = variant_df.loc[variant_df['model'] == first_model, ['question']].copy()
    extracted_numbers = (
        model_df['question']
        .str.findall(number_pattern)
        .explode()
        .dropna()
        .astype(float)
    )
    variant_raw_numbers[variant_name] = extracted_numbers
    integer_mask = np.isclose(extracted_numbers, np.round(extracted_numbers))
    integer_numbers = extracted_numbers[integer_mask]
    fraction_count = int((~integer_mask).sum())
    integer_binned = pd.cut(
        integer_numbers,
        bins=integer_bin_edges,
        labels=raw_integer_bin_labels,
        right=False,
        include_lowest=True,
    )
    integer_counts = integer_binned.value_counts().reindex(raw_integer_bin_labels, fill_value=0)
    counts = pd.Series(
        [fraction_count] + list(integer_counts.iloc[:5].values) + list(integer_counts.iloc[5:].values),
        index=bin_labels,
        dtype=int,
    )
    variant_number_counts[variant_name] = counts
counts_df = pd.DataFrame(variant_number_counts).fillna(0).astype(int)
counts_df

In [ ]:
plot_bin_positions = np.arange(len(bin_labels))
plot_bin_centers = plot_bin_positions + 0.5
variants = list(variant_number_counts)
n_variants = len(variants)
variant_colors = {
    'gsm8k': 'mediumslateblue',
    'main': 'darksalmon',
}
fig, (ax_count, ax_cum) = plt.subplots(2, 1, figsize=(10, 7))
total_bar_width = 0.8
bar_width = total_bar_width / max(n_variants, 1)
bar_offset = (1.0 - total_bar_width) / 2.0
for idx, variant_name in enumerate(variants):
    counts = variant_number_counts[variant_name]
    total_count = int(counts.sum())
    if total_count > 0:
        percentages = counts / total_count * 100.0
    else:
        percentages = counts.astype(float)
    bar_positions = plot_bin_positions + bar_offset + idx * bar_width
    color = variant_colors.get(variant_name.lower(), None)
    ax_count.bar(
        bar_positions,
        percentages.values,
        width=bar_width,
        align='edge',
        alpha=0.8,
        edgecolor='white',
        linewidth=0.8,
        label=variant_name,
        color=color,
    )
    raw_numbers = variant_raw_numbers[variant_name]
    if len(raw_numbers) > 0:
        unique_values, value_counts = np.unique(raw_numbers.values, return_counts=True)
        low_mask = unique_values <= 100
        tail_count = int(value_counts[~low_mask].sum())
        x_values = list(unique_values[low_mask])
        y_counts = list(value_counts[low_mask])
        if tail_count > 0:
            x_values.append(101.0)
            y_counts.append(tail_count)
        x_values = np.array(x_values, dtype=float)
        y_counts = np.array(y_counts, dtype=float)
        cumulative_percentages = np.cumsum(y_counts) / y_counts.sum() * 100.0
        ax_cum.plot(x_values, cumulative_percentages,
                    marker='.', lw=1.0, label=variant_name, color=color)

ax_count.set_ylabel('percent of variant total')
ax_count.set_xticks(plot_bin_centers)
ax_count.set_xticklabels(bin_labels)
ax_count.set_xlim(0, len(bin_labels))
ax_count.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=0))
ax_count.legend(title='Dataset variant')
ax_cum.set_ylabel('cumulative percent')
ax_cum.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=0))
ax_cum.legend(title='Dataset variant')
ax_count.set_xlabel('Extracted number buckets')
ax_cum.set_xlabel('Extracted numbers (all >100 grouped at 101)')
fig.tight_layout()
